In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BorsaBot — 02 | Feature Engineering (Özellik Mühendisliği)
# ═══════════════════════════════════════════════════════════════════
# Bu notebook 01_data_download.py'dan sonra çalıştırılır.
# `ohlcv` dict'i ve `CFG` mevcut olmalı.

In [ ]:
!pip install -q scikit-learn statsmodels

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
# ─────────────────────────────────────────────────────────────────
def get_weights_ffd(d: float, thres: float = 1e-5) -> np.ndarray:
    """FFD weights: log-price memory'yi korur, stasyonerlik sağlar."""
    w = [1.0]
    k = 1
    while abs(w[-1]) >= thres:
        w.append(-w[-1] * (d - k + 1) / k)
        k += 1
    return np.array(w[::-1])

def frac_diff_ffd(series: pd.Series, d: float = 0.4,
                  thres: float = 1e-5) -> pd.Series:
    """Sabit-pencere FFD uygular."""
    w     = get_weights_ffd(d, thres)
    width = len(w)
    out   = {}
    for i in range(width - 1, len(series)):
        chunk = series.iloc[i - width + 1: i + 1].values
        out[series.index[i]] = float(np.dot(w, chunk))
    return pd.Series(out, name=f"fd_{d}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
def compute_features(df: pd.DataFrame, d: float = 0.4) -> pd.DataFrame:
    """
    Ham OHLCV'den özellik matrisi üretir.

    Döner (NaN satırları temizlenmiş):
      fd_close      → frac-diff log kapanış (stasyoner)
      ret_1..ret_24 → gecikmeli getiriler
      vol_24        → 24-bar gerçekleşmiş vol
      vol_72        → 72-bar gerçekleşmiş vol
      rsi_14        → RSI(14)
      atr_14        → ATR(14) / close  (normalize)
      obv_norm      → Normalize OBV
      bb_width      → Bollinger Band genişliği
      close_vwap    → (close - vwap) / std(close, 24)
      log_vol_24    → log(hacim / MA(hacim,24))
    """
    feat = pd.DataFrame(index=df.index)

    # ── Frac diff log kapanış ──────────────────────────────────────
    log_close      = np.log(df["close"])
    feat["fd_close"] = frac_diff_ffd(log_close, d=d)

    # ── Gecikmeli getiriler ────────────────────────────────────────
    ret = df["close"].pct_change()
    for lag in [1, 2, 3, 6, 12, 24]:
        feat[f"ret_{lag}"] = ret.shift(lag)

    # ── Gerçekleşmiş volatilite ────────────────────────────────────
    feat["vol_24"] = ret.rolling(24).std()
    feat["vol_72"] = ret.rolling(72).std()

    # ── RSI(14) ───────────────────────────────────────────────────
    delta = df["close"].diff()
    gain  = delta.clip(lower=0).ewm(com=13, min_periods=14).mean()
    loss  = (-delta.clip(upper=0)).ewm(com=13, min_periods=14).mean()
    feat["rsi_14"] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    # ── ATR(14) ───────────────────────────────────────────────────
    hl  = df["high"] - df["low"]
    hpc = (df["high"] - df["close"].shift(1)).abs()
    lpc = (df["low"]  - df["close"].shift(1)).abs()
    tr  = pd.concat([hl, hpc, lpc], axis=1).max(axis=1)
    feat["atr_14"] = tr.ewm(com=13, min_periods=14).mean() / df["close"]

    # ── OBV (normalize) ───────────────────────────────────────────
    direction = np.sign(df["close"].diff().fillna(0))
    obv = (df["volume"] * direction).cumsum()
    feat["obv_norm"] = (obv - obv.rolling(72).mean()) / (obv.rolling(72).std() + 1e-9)

    # ── Bollinger Band genişliği ───────────────────────────────────
    ma20     = df["close"].rolling(20).mean()
    std20    = df["close"].rolling(20).std()
    feat["bb_width"] = (2 * std20) / (ma20 + 1e-9)

    # ── VWAP sapması ──────────────────────────────────────────────
    typical   = (df["high"] + df["low"] + df["close"]) / 3
    cum_vol   = df["volume"].rolling(24).sum()
    cum_tp_v  = (typical * df["volume"]).rolling(24).sum()
    vwap      = cum_tp_v / (cum_vol + 1e-9)
    feat["close_vwap"] = (df["close"] - vwap) / (df["close"].rolling(24).std() + 1e-9)

    # ── Log hacim ─────────────────────────────────────────────────
    vol_ma = df["volume"].rolling(24).mean()
    feat["log_vol_24"] = np.log(df["volume"] / (vol_ma + 1e-9) + 1e-9)

    return feat.dropna()

In [ ]:
# ─────────────────────────────────────────────────────────────────
features: dict[str, pd.DataFrame] = {}

for sym, df in ohlcv.items():
    print(f"[{sym}] Özellikler hesaplanıyor ...")
    feat = compute_features(df)
    feat.to_parquet(f"{CFG['out_dir']}/{sym}_features.parquet")
    features[sym] = feat
    print(f"  → {feat.shape[0]:,} bar x {feat.shape[1]} özellik")
    print(f"  → Sütunar: {list(feat.columns)}")

print("\n✓ Özellik mühendisliği tamamlandı")